# CS 450 L06: The LLM remembers nothing

This notebook runs a small LLM (Meta's Llama 3.2, 1 billion parameters) right here in Colab.
There is no chat app around it. **You** are the chat system: you decide what text gets sent.

Three experiments:
1. Ask it what year it is.
2. Talk to it twice, with no history.
3. Talk to it twice, sending the history yourself.

Before each experiment, write down your prediction. Then run the cell.

*Runtime: the free CPU runtime is fine. Step 1 takes about two minutes.*

## Step 1. Put the LLM on this computer

This installs Ollama (a program that runs LLMs), starts it, and downloads the model file (about 1.3 GB).
You do not need to read this code.

In [ ]:
import subprocess, time

!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull llama3.2:1b 2>&1 | tail -1

## Step 2. One function that sends text to the LLM

`ask` sends a list of messages to the LLM and returns its reply. Each message has a **role**
(`user` or `assistant`) and its **content**. Nothing else is sent: no memory, no date, no hidden instructions from us.

In [ ]:
import requests

HOST = "http://localhost:11434"
MODEL = "llama3.2:1b"

def ask(messages):
    r = requests.post(f"{HOST}/api/chat",
                      json={"model": MODEL, "messages": messages, "stream": False})
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

print(ask([{"role": "user", "content": "Say hello in five words."}]))

## Experiment 1. What year is it?

**Predict:** today is in 2026. What year will the LLM say?

In [ ]:
for run in range(3):
    print(run + 1, ask([{"role": "user", "content": "What year is it? Answer with just the year."}]))

**Question 1:** Was it right? The model's parameters stopped changing when training ended.
Where could it have learned what year it is?

Some chat apps get this right. Why? Hold that question for next lecture.

## Experiment 2. Two messages, no history

First Sam introduces themself. Then, in a **separate** call, Sam asks what they just said.

**Predict:** what will the second reply say?

In [ ]:
first = "Hi, I'm Sam. I have a biology quiz on cells tomorrow."

reply1 = ask([{"role": "user", "content": first}])
print("SAM:  ", first)
print("LLM:  ", reply1)

print()
print("SAM:   What did I just say?")
print("LLM:  ", ask([{"role": "user", "content": "What did I just say?"}]))

**Question 2:** Read the second `ask(...)` call. What text did the LLM actually receive?
Could any LLM have answered correctly from that text alone?

## Experiment 3. Same question, and this time you send the history

Same LLM, same question. The only change: the list now includes Sam's first message and the LLM's first reply.

**Predict:** what will the reply say now?

In [ ]:
history = [
    {"role": "user", "content": first},
    {"role": "assistant", "content": reply1},
    {"role": "user", "content": "What did I just say?"},
]

for message in history:
    print(message["role"].upper(), ":", message["content"])

print()
print("LLM:  ", ask(history))

**Question 3:** Nothing about the LLM changed between Experiment 2 and Experiment 3.
In one sentence: what changed, and who changed it?

**Question 4:** In a chat app like the one Sam uses, what part of the system does the job you just did by hand?